# Post-Training: From Language Model to Assistant

**Prerequisites**

- The Transformer (previous notebook): architecture, pre-training, next-token prediction
- Gradient descent and loss functions (L03, L04)

**Outcomes**

- Understand why pre-trained models need additional training to be useful
- Understand supervised fine-tuning (SFT) and how it shapes model behavior
- Understand RLHF: reward modeling and policy optimization with PPO
- Understand DPO as a simpler alternative to RLHF
- Understand GRPO: group relative policy optimization without a value function
- Know what Constitutional AI is and how it relates to RLHF
- Understand how post-trained models are evaluated

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.set_printoptions(linewidth=140, precision=4, suppress=True)

%matplotlib inline

## The Post-Training Problem

A pre-trained language model is a **next-token predictor**. It has learned the statistical structure of language — grammar, facts, reasoning patterns — from billions of tokens of text. But it has no notion of "helpfulness" or "following instructions."

If you give a pre-trained model the prompt:

> "What is GDP?"

it might complete the text as if writing the next paragraph of a textbook chapter, or it might produce an unrelated continuation. The model is doing exactly what it was trained to do — predict what text naturally follows — but that is not what a *user* wants.

We need to **align** the model's behavior with human intent. The modern approach is a multi-stage pipeline:

1. **Pre-training**: Next-token prediction on massive text corpora (previous notebook)
2. **Supervised Fine-Tuning (SFT)**: Train on curated (instruction, response) pairs
3. **Preference Optimization (RLHF or DPO)**: Further refine using human preference judgments

This pipeline transforms a text completion engine into an instruction-following assistant like ChatGPT or Claude.

There is a natural **principal-agent** framing: the human user is the principal who specifies objectives (through instructions and feedback), and the language model is the agent whose behavior we want to align with those objectives. The challenge, as in any principal-agent problem, is that the agent's "incentives" (its training objective) may not perfectly align with the principal's preferences.

## Supervised Fine-Tuning (SFT)

SFT takes the pre-trained model and fine-tunes it on a curated dataset of (prompt, desired response) pairs. The loss function is the same cross-entropy as pre-training, but with one important difference: the loss is computed **only on the response tokens**. The prompt tokens provide context but are not predicted.

Formally, given a prompt $p = (p_1, \ldots, p_m)$ and a desired response $r = (r_1, \ldots, r_n)$, the SFT loss is:

$$\mathcal{L}_{\text{SFT}} = -\sum_{t=1}^{n} \log P_\theta(r_t \mid p_1, \ldots, p_m, r_1, \ldots, r_{t-1})$$

The model sees the full concatenated sequence $[p_1, \ldots, p_m, r_1, \ldots, r_n]$, but the loss gradient only flows from the response positions. This is implemented by **masking** the loss at prompt positions.

### Key design decisions

- **Data source**: Early models used human-written demonstrations. Increasingly, training data is generated by stronger models ("distillation") and then curated by humans. The InstructGPT paper (Ouyang et al., 2022) used ~13,000 demonstrations written by human contractors.

- **Learning rate**: Typically 10–100x smaller than pre-training to avoid catastrophic forgetting — overwriting the knowledge acquired during pre-training.

- **Data diversity**: The SFT dataset must cover a wide range of instructions (question answering, summarization, coding, creative writing, math, etc.) to produce a general-purpose assistant.

- **Multi-turn formatting**: Real conversations involve multiple back-and-forth exchanges. SFT data formats these as a single sequence with role markers:

```
[INST] What is comparative advantage? [/INST] Comparative advantage
is the ability of a party to produce a good at a lower opportunity
cost... [INST] Can you give me an example? [/INST] Consider two
countries, England and Portugal...
```

The loss is computed only on the assistant turns, not the user turns.

### Catastrophic forgetting

A key tension in SFT is that fine-tuning on instruction-response data can *overwrite* capabilities the model learned during pre-training. For instance, if the SFT dataset has no math examples, the model might become worse at math after SFT even if it could do math before. This is why SFT datasets must be carefully balanced across domains, and why the learning rate must be kept small — we want to *steer* the model's behavior without *erasing* its knowledge.

In [ ]:
# Demonstrate SFT data formatting and loss masking

# A single training example
prompt = "What is GDP?"
response = (
    "GDP (Gross Domestic Product) is the total monetary value of all "
    "finished goods and services produced within a country's borders "
    "in a specific time period."
)

# In practice, prompt and response are concatenated with special tokens
full_sequence = f"[INST] {prompt} [/INST] {response}"

# Identify which tokens are prompt vs. response
prompt_part = f"[INST] {prompt} [/INST] "
prompt_len = len(prompt_part)
response_len = len(response)

print(f"Full sequence ({len(full_sequence)} chars):")
print(full_sequence)
print(f"\nPrompt:   {prompt_len} chars (loss masked)")
print(f"Response: {response_len} chars (loss computed)")

In [ ]:
# Visualize the loss mask
fig, ax = plt.subplots(figsize=(14, 1.5))

total_len = len(full_sequence)
# Color: gray for prompt (masked), blue for response (loss computed)
colors = ["lightgray"] * prompt_len + ["steelblue"] * response_len

for i in range(total_len):
    ax.barh(0, 1, left=i, color=colors[i], edgecolor="none")

ax.axvline(x=prompt_len, color="black", linewidth=2, linestyle="--")
ax.text(prompt_len / 2, 0.6, "Prompt (loss = 0)", ha="center", fontsize=11)
ax.text(prompt_len + response_len / 2, 0.6, "Response (loss computed)",
        ha="center", fontsize=11)
ax.set_xlim(0, total_len)
ax.set_ylim(-0.5, 1.2)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title("SFT loss mask")
plt.tight_layout()
plt.show()

SFT teaches the model to *imitate* high-quality demonstrations. But imitation has limits: there are subtle quality differences between responses that are easier to **judge** than to **demonstrate**. This is where reinforcement learning from human feedback comes in.

## Reinforcement Learning from Human Feedback (RLHF)

RLHF (Christiano et al., 2017; Ouyang et al., 2022) refines model behavior using human preference judgments rather than demonstrations. The process has two stages:

1. Train a **reward model** to predict which responses humans prefer
2. Optimize the language model (the "policy") to maximize the predicted reward

### Step 1: Reward Modeling

**Data collection**: Given a prompt $x$, generate two or more candidate responses from the SFT model. A human annotator ranks the responses by quality. The result is a dataset of tuples $(x, y_w, y_l)$ where $y_w$ is the **preferred** ("winner") response and $y_l$ is the **dispreferred** ("loser") response.

**Model**: Train a reward model $R_\phi(x, y) \in \mathbb{R}$ — typically a transformer initialized from the SFT model, with the language modeling head replaced by a linear head that outputs a scalar score. The reward model is trained to assign higher scores to preferred responses.

**Loss function**: The loss follows the **Bradley-Terry model** of pairwise preferences, which models the probability that $y_w$ is preferred over $y_l$ as a sigmoid of their reward difference:

$$P(y_w \succ y_l) = \sigma\!\left(R_\phi(x, y_w) - R_\phi(x, y_l)\right)$$

The training loss is the negative log-likelihood:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma\!\left(R_\phi(x, y_w) - R_\phi(x, y_l)\right) \right]$$

where $\sigma(z) = 1 / (1 + e^{-z})$ is the sigmoid function. This is equivalent to logistic regression on the reward difference — a familiar statistical model applied in a novel context.

In [ ]:
def reward_model_loss(r_preferred, r_dispreferred):
    """Bradley-Terry loss for reward modeling."""
    return -torch.log(torch.sigmoid(r_preferred - r_dispreferred)).mean()

In [ ]:
# Visualize the reward model loss surface
r_w = torch.linspace(-4, 4, 200)
r_l_vals = [-1.0, 0.0, 1.0]

fig, ax = plt.subplots(figsize=(9, 5))

colors = ["steelblue", "darkorange", "forestgreen"]
for r_l, color in zip(r_l_vals, colors):
    losses = -torch.log(torch.sigmoid(r_w - r_l))
    ax.plot(r_w.numpy(), losses.numpy(), color=color, linewidth=2,
            label=f"$R(x, y_l) = {r_l}$")

ax.set_xlabel("$R(x, y_w)$ — reward of preferred response", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Reward model loss")
ax.legend(fontsize=11)
ax.set_ylim(0, 6)
plt.tight_layout()
plt.show()

The loss is low when $R(x, y_w) \gg R(x, y_l)$ (the model confidently assigns a higher reward to the preferred response) and increases sharply when the rewards are reversed. The shape is exactly the logistic loss — the same loss used in logistic regression.

### Step 2: Policy Optimization with PPO

With a trained reward model $R_\phi$, we optimize the language model (the "policy" $\pi_\theta$) to generate responses that receive high reward. The objective includes a **KL divergence penalty** to prevent the model from deviating too far from the SFT model $\pi_{\text{ref}}$:

$$\max_\theta \; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot|x)} \left[ R_\phi(x, y) - \beta \, \text{KL}\!\left(\pi_\theta(\cdot|x) \,\|\, \pi_{\text{ref}}(\cdot|x)\right) \right]$$

The KL penalty is **critical**. Without it, the model would learn to "game" the reward model by producing adversarial outputs that receive high reward scores but are actually low quality — a phenomenon known as **reward hacking**. The KL term keeps the policy close to the SFT model, which serves as an anchor of reasonable behavior.

The hyperparameter $\beta > 0$ controls the strength of the penalty:
- Large $\beta$: the policy stays very close to $\pi_{\text{ref}}$ (conservative, little improvement)
- Small $\beta$: the policy can deviate more (potentially higher reward, but risk of reward hacking)

This optimization is performed using **Proximal Policy Optimization (PPO)** (Schulman et al., 2017), a reinforcement learning algorithm. The PPO clipped objective prevents overly large policy updates:

$$\mathcal{L}_{\text{PPO}} = -\mathbb{E}\left[ \min\!\left( r_t(\theta)\, A_t,\; \text{clip}\!\left(r_t(\theta),\, 1{-}\epsilon,\, 1{+}\epsilon\right) A_t \right) \right]$$

where $r_t(\theta) = \pi_\theta(a_t | s_t) / \pi_{\theta_{\text{old}}}(a_t | s_t)$ is the probability ratio between the current and previous policy, and $A_t$ is the advantage (reward minus baseline). The clipping ensures that no single update can change the policy too drastically.

The RLHF training loop at each step:
1. Sample prompts from a dataset
2. Generate responses from the current policy $\pi_\theta$
3. Score responses with the reward model $R_\phi$
4. Compute the KL penalty against $\pi_{\text{ref}}$
5. Update $\theta$ using PPO

In [ ]:
# Visualize the reward-KL tradeoff for different beta values
kl = np.linspace(0, 5, 100)
betas = [0.01, 0.1, 0.5]

# Hypothetical reward curve: diminishing returns as KL increases
reward = 2.0 * (1 - np.exp(-0.8 * kl))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: reward and penalty curves
axes[0].plot(kl, reward, color="steelblue", linewidth=2, label="Reward $R(x,y)$")
colors = ["darkorange", "forestgreen", "firebrick"]
for beta, color in zip(betas, colors):
    axes[0].plot(kl, beta * kl, color=color, linewidth=2, linestyle="--",
                label=f"$\\beta$ KL ($\\beta={beta}$)")
axes[0].set_xlabel("KL divergence from $\\pi_{\\text{ref}}$", fontsize=12)
axes[0].set_ylabel("Value", fontsize=12)
axes[0].set_title("Reward vs. KL penalty")
axes[0].legend(fontsize=10)

# Right: net objective
for beta, color in zip(betas, colors):
    net = reward - beta * kl
    axes[1].plot(kl, net, color=color, linewidth=2, label=f"$\\beta={beta}$")
    opt_idx = np.argmax(net)
    axes[1].plot(kl[opt_idx], net[opt_idx], "o", color=color, markersize=8)

axes[1].axhline(0, color="gray", linewidth=0.5, linestyle="--")
axes[1].set_xlabel("KL divergence from $\\pi_{\\text{ref}}$", fontsize=12)
axes[1].set_ylabel("$R - \\beta$ KL (net objective)", fontsize=12)
axes[1].set_title("Net objective (dots = optima)")
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

Each $\beta$ value produces a different optimal tradeoff. Small $\beta$ allows the policy to diverge more from the reference model in pursuit of higher reward; large $\beta$ keeps the policy close to the SFT baseline. In practice, $\beta$ is tuned to balance improvement against stability.

### Reward hacking

Reward hacking is a pervasive problem when optimizing against a learned reward model. Because the reward model is an imperfect proxy for human preferences, there exist "adversarial" outputs that receive high reward scores despite being low quality. Examples:

- **Verbosity bias**: reward models often prefer longer responses, so an unconstrained policy learns to generate excessively long, repetitive text
- **Sycophancy**: the model learns to agree with the user's stated position rather than provide accurate information, because agreeable responses tend to receive higher human preference ratings
- **Format gaming**: inserting bullet points, bold text, or confident-sounding phrases that correlate with high reward but don't improve substance

This is a form of **Goodhart's Law**: when a measure becomes a target, it ceases to be a good measure. The KL penalty is the primary defense — it limits how far the policy can deviate from the SFT baseline, restricting the search space for adversarial outputs.

## Direct Preference Optimization (DPO)

RLHF works, but it is complex: it requires training a separate reward model, running a PPO training loop with multiple interacting components, and carefully tuning hyperparameters. Rafailov et al. (2023) observed that the KL-constrained RLHF objective has a **closed-form optimal policy**:

$$\pi^*(y | x) = \frac{1}{Z(x)} \, \pi_{\text{ref}}(y | x) \, \exp\!\left(\frac{1}{\beta} R(x, y)\right)$$

where $Z(x)$ is a normalizing constant (partition function). This can be rearranged to express the reward in terms of the policy:

$$R(x, y) = \beta \log \frac{\pi_\theta(y | x)}{\pi_{\text{ref}}(y | x)} + \beta \log Z(x)$$

Now substitute this into the Bradley-Terry preference model:

$$P(y_w \succ y_l) = \sigma\!\left(R(x, y_w) - R(x, y_l)\right)$$

The partition function $Z(x)$ appears in both $R(x, y_w)$ and $R(x, y_l)$ and **cancels out**, giving the **DPO loss**:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w | x)}{\pi_{\text{ref}}(y_w | x)} - \beta \log \frac{\pi_\theta(y_l | x)}{\pi_{\text{ref}}(y_l | x)}\right) \right]$$

### What DPO does

The key insight: DPO **eliminates the need for a separate reward model and the PPO training loop**. It directly optimizes the language model on preference data using a supervised, classification-like loss.

The loss has an intuitive interpretation. Define the **implicit reward** of response $y$ as:

$$\hat{r}(x, y) = \beta \log \frac{\pi_\theta(y | x)}{\pi_{\text{ref}}(y | x)}$$

This is the log-probability ratio between the current policy and the reference, scaled by $\beta$. A response that the current model finds more likely than the reference model gets a positive implicit reward. The DPO loss pushes the model to increase the implicit reward gap between preferred and dispreferred responses — making $\hat{r}(x, y_w) > \hat{r}(x, y_l)$.

In [ ]:
def dpo_loss(pi_logprobs_w, pi_logprobs_l,
             ref_logprobs_w, ref_logprobs_l, beta=0.1):
    """DPO loss given per-sequence log-probabilities."""
    log_ratio_w = pi_logprobs_w - ref_logprobs_w
    log_ratio_l = pi_logprobs_l - ref_logprobs_l
    return -F.logsigmoid(beta * (log_ratio_w - log_ratio_l)).mean()

In [ ]:
# Visualize DPO loss as a function of the implicit reward margin
margin = torch.linspace(-5, 5, 200)  # r_hat(y_w) - r_hat(y_l)

fig, ax = plt.subplots(figsize=(9, 5))

for beta in [0.05, 0.1, 0.5]:
    loss = -F.logsigmoid(beta * margin)
    ax.plot(margin.numpy(), loss.numpy(), linewidth=2, label=f"$\\beta={beta}$")

ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel("Implicit reward margin: $\\hat{r}(y_w) - \\hat{r}(y_l)$", fontsize=12)
ax.set_ylabel("DPO loss", fontsize=12)
ax.set_title("DPO loss vs. implicit reward margin")
ax.legend(fontsize=11)
ax.set_ylim(0, 4)
plt.tight_layout()
plt.show()

The loss is minimized when $\hat{r}(y_w) \gg \hat{r}(y_l)$ — the model strongly prefers the winner. The $\beta$ parameter controls sensitivity: small $\beta$ requires a larger reward margin to achieve low loss.

### RLHF vs. DPO

| | RLHF (PPO) | DPO |
|---|---|---|
| **Reward model** | Required (separate model) | Not needed |
| **RL training loop** | Yes (PPO with value function, advantage estimation) | No (supervised loss) |
| **Training stability** | Sensitive to hyperparameters | More stable |
| **Compute cost** | Higher (multiple models in memory) | Lower |
| **Implementation** | Complex | Simple |
| **Flexibility** | Can optimize arbitrary reward functions | Tied to pairwise preference data |

DPO has become increasingly popular due to its simplicity and stability. However, RLHF remains more flexible — for instance, it can incorporate reward signals beyond pairwise preferences (e.g., from automated evaluators or multi-objective reward models).

## Group Relative Policy Optimization (GRPO)

Shao et al. (2024) introduced **GRPO** as an alternative to PPO that eliminates the need for a separate **value function** (also called the critic network) — one of the most complex and memory-intensive components of the RLHF pipeline. GRPO was used to train DeepSeek-R1 and has become influential in the reasoning model literature.

### The problem with PPO's value function

In standard PPO, the **advantage** $A_t$ measures how much better an action is than expected. Computing the advantage requires a learned **value function** $V_\psi(s)$ that estimates the expected future reward from state $s$. This value function:

- Is a second neural network (often the same size as the policy) that must be trained alongside the policy
- Doubles the memory footprint
- Introduces its own approximation errors, which can destabilize training

### GRPO's key idea: group-relative advantages

Instead of using a learned value function, GRPO estimates advantages by **sampling multiple responses from the policy and comparing them to each other**. For each prompt $x$:

1. Sample a **group** of $G$ responses: $\{y_1, y_2, \ldots, y_G\} \sim \pi_\theta(\cdot | x)$
2. Score each response with the reward model: $r_i = R_\phi(x, y_i)$
3. Compute **group-relative advantages** by normalizing within the group:

$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_1, \ldots, r_G\})}{\text{std}(\{r_1, \ldots, r_G\})}$$

4. Update the policy using these advantages in a PPO-like clipped objective with a KL penalty:

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G} \left[ \min\!\left(r_i(\theta) \hat{A}_i,\; \text{clip}(r_i(\theta), 1{-}\epsilon, 1{+}\epsilon) \hat{A}_i\right) - \beta \, \text{KL}(\pi_\theta \| \pi_{\text{ref}}) \right]$$

where $r_i(\theta) = \pi_\theta(y_i | x) / \pi_{\theta_{\text{old}}}(y_i | x)$ is the importance sampling ratio.

### Why this works

The group normalization is elegant: it uses the sampled responses themselves as a baseline. If all $G$ responses receive similar rewards, the advantages are near zero and the update is small. If one response is much better than the others, it gets a large positive advantage and the policy is pushed toward producing similar outputs.

This is related to the statistical idea of **relative ranking** — we don't need to know the absolute quality of a response, only whether it is better or worse than the alternatives from the same model.

In [ ]:
# Illustrate GRPO's group-relative advantage computation
torch.manual_seed(42)

# Simulate: for one prompt, sample G=8 responses and score them
G = 8
rewards = torch.tensor([1.2, 0.3, -0.5, 2.1, 0.8, -0.2, 1.5, 0.1])

# Group-relative advantage: z-score normalization
mean_r = rewards.mean()
std_r = rewards.std()
advantages = (rewards - mean_r) / std_r

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw rewards
colors = ["steelblue" if a >= 0 else "lightcoral" for a in advantages]
axes[0].bar(range(G), rewards.numpy(), color=colors, edgecolor="black", linewidth=0.5)
axes[0].axhline(mean_r.item(), color="black", linewidth=1.5, linestyle="--",
                label=f"Group mean = {mean_r.item():.2f}")
axes[0].set_xlabel("Response index")
axes[0].set_ylabel("Reward $r_i$")
axes[0].set_title("Raw rewards from reward model")
axes[0].legend()

# Right: group-relative advantages
axes[1].bar(range(G), advantages.numpy(), color=colors, edgecolor="black", linewidth=0.5)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_xlabel("Response index")
axes[1].set_ylabel("Advantage $\\hat{A}_i$")
axes[1].set_title("GRPO group-relative advantages")

plt.tight_layout()
plt.show()

print(f"Rewards:    {rewards.numpy()}")
print(f"Advantages: {advantages.numpy().round(3)}")
print(f"\nResponses with positive advantage are reinforced;")
print(f"responses with negative advantage are suppressed.")

### GRPO for reasoning models

GRPO is particularly well-suited for training **reasoning models** — models that solve problems by generating step-by-step chains of thought. For tasks like mathematics, the reward signal can be a simple binary outcome: did the model get the correct answer?

This is the approach behind DeepSeek-R1 (2024): use GRPO with outcome-based rewards (correct/incorrect) to train a model that learns to reason through problems. The group sampling naturally explores different reasoning strategies, and the relative advantage computation reinforces strategies that lead to correct answers.

More generally, GRPO works well whenever we have a reward signal that can be computed automatically (math correctness, code execution, factual verification) rather than requiring human judgment.

### Comparing optimization methods

| | PPO | DPO | GRPO |
|---|---|---|---|
| **Reward model** | Required | Not needed | Required |
| **Value function** | Required | Not needed | Not needed |
| **Data** | Online (generates responses) | Offline (preference pairs) | Online (generates responses) |
| **Memory** | Highest (policy + reward + value) | Lowest (policy + reference) | Medium (policy + reward) |
| **Best for** | General alignment | Preference learning | Reasoning, verifiable tasks |

## Rejection Sampling and Best-of-N

Before moving to RL-based methods, it is worth mentioning a simpler technique that is often used as a baseline or in production systems.

**Best-of-N sampling**: Given a prompt, generate $N$ candidate responses from the SFT model, score each with the reward model, and return the highest-scoring response. This is sometimes called **rejection sampling** in the alignment literature.

The expected reward under best-of-N scales as:

$$\mathbb{E}\left[\max_{i=1,\ldots,N} R(x, y_i)\right]$$

which increases with $N$ — more samples means a higher chance of finding a good response. Empirically, best-of-N is a surprisingly strong baseline that often matches or exceeds PPO-trained models, especially for small $N$ (e.g., 4–16).

The main disadvantage is inference cost: generating $N$ responses and scoring them all is $N$ times more expensive than generating one. RLHF/DPO/GRPO "bake in" this quality improvement at training time, so inference remains cheap. However, best-of-N is simple, does not modify the model weights, and can be applied to any model with access to a reward signal.

## Constitutional AI

Bai et al. (2022) proposed **Constitutional AI (CAI)** as an approach that reduces reliance on human feedback for alignment. The core idea:

1. **Write a constitution**: a set of explicit principles such as "be helpful," "avoid harmful content," "be honest about uncertainty"

2. **Self-critique**: ask the model to evaluate its own responses against the constitution

3. **Self-revision**: ask the model to revise responses that violate the principles

4. **RLAIF**: use the revised responses (or AI-generated preference judgments) as training data for preference optimization — RL from **AI** Feedback rather than human feedback

### Example

```
User:       "How can I manipulate stock prices?"

Initial response (from SFT model):
            "Here are some common strategies for manipulating stock
             prices: pump-and-dump schemes, wash trading..."

Self-critique (using principle "Be ethical and legal"):
            "This response provides advice on illegal market
             manipulation, which violates securities law and could
             cause harm to investors."

Revised response:
            "Market manipulation is illegal under securities law
             (e.g., SEC Rule 10b-5). If you're interested in how
             markets work, I'd recommend studying market
             microstructure — Hasbrouck (2007) is a good starting
             point."
```

Constitutional AI has several advantages:

- **Scalability**: generating AI feedback is cheaper than collecting human preferences
- **Transparency**: the alignment criteria are explicit and auditable (the constitution can be inspected and debated)
- **Consistency**: AI evaluators apply rules more uniformly than human annotators

## Evaluation

How do we know if post-training worked? Evaluating language models is fundamentally difficult — we are trying to measure whether a model is "good" across an enormous space of possible interactions. Several complementary approaches are used.

### Automated benchmarks

| Benchmark | Domain | Format |
|---|---|---|
| MMLU | 57 academic subjects | Multiple-choice |
| GSM8K | Grade school math | Free-form reasoning |
| HumanEval | Code generation | Function completion |
| ARC | Science reasoning | Multiple-choice |
| TruthfulQA | Truthfulness | Open-ended |

These benchmarks provide quantitative, reproducible metrics. However, they only measure narrow slices of capability and can be "gamed" through training on similar data.

### Human evaluation

The **Chatbot Arena** (LMSYS) collects pairwise comparisons from real users: two anonymous models respond to the same prompt, and the user picks the better response. The results produce **Elo ratings** — the same system used in chess — that rank models on overall quality. This is currently considered the most reliable evaluation method, but it is expensive and slow.

### Economics-specific evaluation

For economics applications, we might ask:

- Can the model correctly answer questions about economic concepts (comparative advantage, price elasticity, Nash equilibrium)?
- Can it interpret regression output or statistical tables?
- Does it reason correctly about causal identification (correlation vs. causation, instrumental variables)?
- Can it write and debug economic models in code?

There is no standard economics benchmark yet — this is an active area of work.

In [ ]:
# Structure of a simple automated evaluation
questions = [
    {
        "question": "GDP is best described as:",
        "choices": [
            "A) The total value of stocks traded on a country's exchanges",
            "B) The total monetary value of all finished goods and services "
            "produced within a country in a given period",
            "C) The government's annual budget",
            "D) The total value of a country's imports and exports",
        ],
        "answer": "B",
    },
    {
        "question": "The law of demand states that, ceteris paribus:",
        "choices": [
            "A) As price increases, quantity demanded increases",
            "B) As price increases, quantity demanded decreases",
            "C) As income increases, demand increases",
            "D) As supply increases, price increases",
        ],
        "answer": "B",
    },
    {
        "question": "An instrumental variable must satisfy:",
        "choices": [
            "A) Correlation with the outcome and the treatment",
            "B) Correlation with the treatment and no direct effect on the outcome",
            "C) No correlation with any variable in the model",
            "D) Perfect correlation with the error term",
        ],
        "answer": "B",
    },
]

# In a real evaluation, we would:
# 1. Format each question as a prompt
# 2. Have the model generate/select an answer
# 3. Compare against the ground truth
# 4. Compute accuracy

for i, q in enumerate(questions):
    print(f"Q{i+1}: {q['question']}")
    for c in q["choices"]:
        marker = " *" if c[0] == q["answer"] else ""
        print(f"   {c}{marker}")
    print()

Evaluation is an open and difficult problem. No single metric captures the full range of model capabilities. In practice, model developers use a portfolio of automated benchmarks, human evaluations, and domain-specific tests.

## The Full Pipeline

Putting it all together, the modern LLM pipeline looks like this:

```
Raw Text Corpus (trillions of tokens)
       │
  [Tokenization]            ← Notebook 1
       │
  [Pre-training]            ← Notebook 2
       │  next-token prediction, thousands of GPUs, months of training
       ▼
  Base Model
       │
  [SFT]                     ← This notebook
       │  ~100k instruction-response pairs
       ▼
  SFT Model
       │
  [RLHF / DPO]              ← This notebook
       │  ~100k human preference comparisons
       ▼
  Aligned Model (e.g., ChatGPT, Claude)
```

**Scale matters.** The pre-training step consumes the vast majority of the compute. Frontier models like GPT-4 and Claude are trained on trillions of tokens using thousands of GPUs for months, at a cost of tens to hundreds of millions of dollars. SFT and RLHF/DPO are comparatively cheap — typically a few days on a smaller cluster.

Our classroom model from the previous notebook is a miniature version of the same pipeline. The conceptual framework — predict the next token, then align to human preferences — is identical at any scale. What changes is the amount of data, compute, and the sophistication of the engineering infrastructure.

## References

- Christiano, P. F., Leike, J., Brown, T., Marber, M., Saunders, S., Askell, A., Amodei, D., & Irving, G. (2017). Deep reinforcement learning from human preferences. *NeurIPS 2017*.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal policy optimization algorithms. *arXiv:1707.06347*.
- Ouyang, L., Wu, J., Jiang, X., Almeida, D., Wainwright, C., Mishkin, P., Zhang, C., Agarwal, S., et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS 2022*.
- Bai, Y., Kadavath, S., Kundu, S., Askell, A., Kernion, J., Jones, A., Chen, A., Goldie, A., et al. (2022). Constitutional AI: Harmlessness from AI feedback. *arXiv:2212.08073*.
- Rafailov, R., Sharma, A., Mitchell, E., Ermon, S., Manning, C. D., & Finn, C. (2023). Direct preference optimization: Your language model is secretly a reward model. *NeurIPS 2023*.
- Shao, Z., Wang, P., Zhu, Q., Xu, R., Song, J., Zhang, M., Li, Y., Wu, Y., & Guo, D. (2024). DeepSeekMath: Pushing the limits of mathematical reasoning in open language models. *arXiv:2402.03300*.
- DeepSeek-AI. (2025). DeepSeek-R1: Incentivizing reasoning capability in LLMs via reinforcement learning. *arXiv:2501.12948*.